In [2]:
!pip install vllm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.2/433.2 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 148.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 164.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 116.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

In [1]:
import os
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import json
import logging
# ... all other imports ...
import logging
import pandas as pd
import re
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s %(message)s")
log = logging.getLogger("self_play")

def extract_boxed_answer(text: str) -> str:
    """Extracts the math sequence inside \boxed{}"""
    match = re.search(r'\\boxed{([^}]+)}', text)
    if match:
        return match.group(1).strip()
    return None

def format_sharegpt(prompt: str, generated_response: str) -> dict:
    """Format into standard ShareGPT conversational format"""
    return {
        "conversations": [
            {"from": "user", "value": prompt},
            {"from": "assistant", "value": generated_response}
        ]
    }

def main():
    # Configure these paths depending on if you are running locally or on Colab
    csv_path = "train.csv"  # Ensure train.csv is in the working directory
    base_model_path = "hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4" # Update to local path if downloaded via KaggleHub
    lora_dir = "/content" # The clean directory containing adapter_config.json and safetensors
    out_file = "self_play_dataset.jsonl"

    if not os.path.exists(csv_path):
        log.error(f"{csv_path} not found. Please place training data in the active directory.")
        return

    df = pd.read_csv(csv_path)

    # DATASET REFINEMENT: Filter for complex puzzles
    complex_df = df[df['prompt'].str.len() > 250]

    # Sample 10,000 unseen puzzles to attempt
    SAMPLE_SIZE = 10000
    if len(complex_df) > SAMPLE_SIZE:
        sampled_df = complex_df.sample(SAMPLE_SIZE, random_state=42)
    else:
        sampled_df = complex_df

    log.info(f"Loaded {len(sampled_df)} high-complexity puzzles for Rejection Sampling.")

    # The System-Baked formatting structural enforcing prompt
    system_prompt = """You are an elite mathematical logician. Reverse-engineer the puzzle below. I will give you the Prompt. Your job is to output the step-by-step thinking process that logically arrives at the precise answer.
    Strict formatting rules:
    1. Your entire response MUST follow this EXACT wrapping structure:

<think>
[GIVEN] Extract the core facts.
[HYPOTHESIS] Identify the pattern or mathematical theorem to use.
[PROOF] Execute the logic step-by-step.
[VERIFICATION] Check the result using an alternative secondary method.
</think>
\\boxed{...}

    2. Avoid unnecessary rambling. Do exactly NO brute-forcing of large combinations; find the conceptual pattern.
    3. The final answer MUST be wrapped in the LaTeX \\boxed{} command."""

    # Compile the prompts
    prompts = []
    ground_truths = []

    for idx, row in sampled_df.iterrows():
        # Notice we do NOT feed the answer to the model! It must reason it out itself.
        user_prompt = f"PROMPT:\n{row['prompt']}\n\nPlease generate the logical reasoning trace leading to the answer."
        full_text = f"System: {system_prompt}\n\nUser: {user_prompt}\n\nAssistant: "
        prompts.append(full_text)
        ground_truths.append(str(row["answer"]).strip())

    log.info("Loading vLLM Engine. This requires significant VRAM...")

    # Initialize vLLM with LoRA enabled
    try:
            llm = LLM(
        model="hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4", # Example: A 4-bit quantization that easily fits on an 80GB A100!
        tensor_parallel_size=1,
        max_model_len=8192,
        gpu_memory_utilization=0.95
    )


    except Exception as e:
        log.error(f"Failed to load vLLM Engine: {e}")
        return

    # Kaggle evaluation parameters
    sampling_params = SamplingParams(
        temperature=0.0, # Greedy deterministic decoding
        top_p=1.0,
        max_tokens=6000
    )

    log.info("Executing Batch Inference on all puzzles...")

    # Pass the Base Model + LoRA Request dynamically
    outputs = llm.generate(
        prompts,
        sampling_params,
        lora_request=LoRARequest("nemotron_adapter", 1, lora_dir)
    )

    valid_count = 0
    with open(out_file, "w", encoding="utf-8") as f:
        for idx, output in enumerate(outputs):
            generated_text = output.outputs[0].text
            ground_truth = ground_truths[idx]

            # Outcome Validation
            model_answer = extract_boxed_answer(generated_text)

            if model_answer is None:
                log.warning(f"Puzzle {idx}: Model failed to output \\boxed{{}}. Rejecting.")
                continue

            # THE REJECTION SAMPLING FILTER
            if model_answer == ground_truth:
                # The model perfectly reasoned through the unseen puzzle! Save the trace.
                dataset_row = format_sharegpt(prompts[idx], generated_text)
                f.write(json.dumps(dataset_row, ensure_ascii=False) + "\n")
                valid_count += 1
            else:
                # The model hallucinated or math was wrong. Discard the toxic trace.
                pass

    log.info(f"✨ Self-Play Complete! The model successfully solved {valid_count} unseen puzzles.")
    log.info(f"Saved {valid_count} zero-cost pristine traces to {out_file}.")

if __name__ == "__main__":
    main()


INFO 03-23 00:42:58 [utils.py:233] non-default args: {'max_model_len': 8192, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'hugging-quants/Meta-Llama-3.1-70B-Instruct-AWQ-INT4'}
INFO 03-23 00:42:59 [model.py:533] Resolved architecture: LlamaForCausalLM
INFO 03-23 00:42:59 [model.py:1582] Using max model len 8192
INFO 03-23 00:43:00 [awq_marlin.py:162] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 03-23 00:43:00 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.


Parse safetensors files:   0%|          | 0/9 [00:00<?, ?it/s]

INFO 03-23 00:43:00 [vllm.py:754] Asynchronous scheduling is enabled.
INFO 03-23 00:44:40 [llm.py:391] Supported tasks: ['generate']


Rendering prompts:   0%|          | 0/4775 [00:00<?, ?it/s]

ValueError: Got lora_request LoRARequest(lora_name='nemotron_adapter', lora_int_id=1, lora_path='/content', base_model_name=None, tensorizer_config_dict=None, load_inplace=False) but LoRA is not enabled!